# SlowFast Network — Data Exploration + Preprocessing

| Field | Value |
|---|---|
| **Author** | [Your Name] |
| **Model** | SlowFast Network |
| **Task** | Data Exploration + Preprocessing |
| **Dataset** | 855 videos — 531 normal / 324 shoplifting |
| **What I did** | Loaded dataset, visualized stats, standardized frames, sampled Slow (8) / Fast (32), applied augmentation, saved `.npy` tensors. |
| **Problems/Solutions** | Imbalance handled in training. Processed files are large. |

In [ ]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print('✅ All imports successful')

In [ ]:
# ── CELL 2: Configuration ─────────────────────────────────────────────────────

DATA_DIR = Path(r"C:\Users\POTATO\Desktop\Cellula\Shoplifting_Detection\Data")

CLASSES = {0: "non shop lifters", 1: "shop lifters"}

SLOW_FRAMES = 8
FAST_FRAMES = 32
FRAME_SIZE  = (224, 224)
ALPHA       = FAST_FRAMES // SLOW_FRAMES

print(f"📁 Data dir    : {DATA_DIR.resolve()}")
print(f"🐢 Slow frames : {SLOW_FRAMES}")
print(f"⚡ Fast frames : {FAST_FRAMES}")
print(f"📐 Frame size  : {FRAME_SIZE}")
print(f"⚖️  Alpha ratio : {ALPHA}")

In [ ]:
# ── CELL 3: Scan Dataset ──────────────────────────────────────────────────────

video_extensions = {'.mp4', '.avi', '.mov', '.mkv'}
dataset = []

for label, class_name in CLASSES.items():
    class_dir = DATA_DIR / class_name
    if not class_dir.exists():
        print(f"⚠️  Folder not found: {class_dir}")
        continue
    # ⚠️ FIX: sorting the files ensures cross-platform deterministic order!
    videos = sorted([f for f in class_dir.iterdir()
                     if f.suffix.lower() in video_extensions])
    for v in videos:
        dataset.append({"path": v, "label": label, "class": class_name})

print(f"\n📊 Dataset Summary")
print(f"{'─'*35}")
print(f"Total videos : {len(dataset)}")
for label, class_name in CLASSES.items():
    count = sum(1 for d in dataset if d['label'] == label)
    print(f"  {class_name:20s} : {count} videos")

In [ ]:
# ── CELL 4: Explore Video Properties ─────────────────────────────────────────

def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    fps      = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = n_frames / fps if fps > 0 else 0
    cap.release()
    return {"fps": fps, "n_frames": n_frames,
            "width": width, "height": height, "duration": duration}

print("🔍 Scanning video properties...\n")
stats = defaultdict(list)
for item in dataset:
    info = get_video_info(item["path"])
    if info:
        item.update(info)
        for k, v in info.items():
            stats[k].append(v)

print(f"{'Property':<15} {'Min':>8} {'Max':>8} {'Mean':>8}")
print("─" * 42)
for k, v in stats.items():
    print(f"{k:<15} {min(v):>8.1f} {max(v):>8.1f} {np.mean(v):>8.1f}")

In [ ]:
# ── CELL 5: Visualize Distribution ───────────────────────────────────────────

os.makedirs("ml_pipeline/Notebooks/plots", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Dataset Distribution", fontsize=14, fontweight='bold')

class_counts = [sum(1 for d in dataset if d['label'] == l) for l in CLASSES]
axes[0].bar(list(CLASSES.values()), class_counts, color=['steelblue', 'tomato'])
axes[0].set_title("Class Balance")
axes[0].set_ylabel("Number of Videos")
for i, v in enumerate(class_counts):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

axes[1].hist(stats["n_frames"], bins=20, color='mediumseagreen', edgecolor='black')
axes[1].axvline(np.mean(stats["n_frames"]), color='red', linestyle='--',
                label=f'Mean={np.mean(stats["n_frames"]):.0f}')
axes[1].set_title("Frame Count Distribution")
axes[1].set_xlabel("Number of Frames")
axes[1].legend()

axes[2].hist(stats["duration"], bins=20, color='mediumpurple', edgecolor='black')
axes[2].axvline(np.mean(stats["duration"]), color='red', linestyle='--',
                label=f'Mean={np.mean(stats["duration"]):.1f}s')
axes[2].set_title("Video Duration Distribution")
axes[2].set_xlabel("Duration (seconds)")
axes[2].legend()

plt.tight_layout()
plt.savefig("ml_pipeline/Notebooks/plots/dataset_distribution.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved")

In [ ]:
# ── CELL 6: Sample Frames Visualization ──────────────────────────────────────

def extract_sample_frames(video_path, n=6):
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, n, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
    cap.release()
    return frames

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle("Sample Frames — Top: Normal | Bottom: Shoplifting",
             fontsize=13, fontweight='bold')

for row, (label, class_name) in enumerate(CLASSES.items()):
    sample = random.choice([d for d in dataset if d['label'] == label])
    frames = extract_sample_frames(sample["path"], n=6)
    for col, frame in enumerate(frames):
        if col < len(frames):
            axes[row, col].imshow(frames[col])
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(class_name, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("ml_pipeline/Notebooks/plots/sample_frames.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample frames saved")

In [ ]:
# ── CELL 7: Frame Sampling + Preprocessing ───────────────────────────────────

def sample_frames(video_path, n_frames):
    cap    = cv2.VideoCapture(str(video_path))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, FRAME_SIZE)
            frames.append(frame)
        else:
            if frames:
                frames.append(frames[-1])
    cap.release()
    while len(frames) < n_frames:
        if frames:
            frames.append(frames[-1])
        else:
            frames.append(np.zeros((*FRAME_SIZE, 3), dtype=np.uint8))
    return np.array(frames[:n_frames])

def normalize(frames):
    return frames.astype(np.float32) / 255.0

def preprocess_video(video_path):
    slow = normalize(sample_frames(video_path, SLOW_FRAMES))
    fast = normalize(sample_frames(video_path, FAST_FRAMES))
    return slow, fast

# Test on one video
test_video = dataset[0]
slow, fast = preprocess_video(test_video["path"])
print(f"✅ Preprocessing test passed!")
print(f"   Slow shape : {slow.shape}")
print(f"   Fast shape : {fast.shape}")
print(f"   Pixel range: [{slow.min():.2f}, {slow.max():.2f}]")

In [ ]:
# ── CELL 8: Augmentation ──────────────────────────────────────────────────────

def augment_video(frames):
    if random.random() > 0.5:
        frames = frames[:, :, ::-1, :]
    if random.random() > 0.5:
        factor = random.uniform(0.6, 1.4)
        frames = np.clip(frames * factor, 0, 1)
    if random.random() > 0.5:
        mean   = frames.mean()
        factor = random.uniform(0.7, 1.3)
        frames = np.clip((frames - mean) * factor + mean, 0, 1)
    return frames.astype(np.float32)

slow_aug = augment_video(slow.copy())
print(f"✅ Augmentation test passed! Shape: {slow_aug.shape}")

In [ ]:
# ── CELL 9: Visualize Augmentation ───────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle("Original (top) vs Augmented (bottom) — Slow Pathway", fontsize=13)

for i in range(4):
    axes[0, i].imshow(slow[i])
    axes[0, i].set_title(f"Original {i+1}")
    axes[0, i].axis('off')
    axes[1, i].imshow(slow_aug[i])
    axes[1, i].set_title(f"Augmented {i+1}")
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig("ml_pipeline/Notebooks/plots/augmentation_comparison.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Augmentation plot saved")

In [ ]:
# ── CELL 10: Process Full Dataset ────────────────────────────────────────────

SAVE_DIR = Path(r"C:\Users\POTATO\Desktop\Cellula\Shoplifting_Detection\Data\processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

slow_all, fast_all, labels_all, filenames = [], [], [], []
failed = []

print(f"🔄 Processing {len(dataset)} videos...")
for i, item in enumerate(dataset):
    try:
        s, f = preprocess_video(item["path"])
        slow_all.append(s)
        fast_all.append(f)
        labels_all.append(item["label"])
        filenames.append(item["path"].name)
        if (i + 1) % 10 == 0:
            print(f"   [{i+1}/{len(dataset)}] done...")
    except Exception as e:
        print(f"   ⚠️  Failed: {item['path'].name} → {e}")
        failed.append(item["path"].name)

slow_all   = np.array(slow_all)
fast_all   = np.array(fast_all)
labels_all = np.array(labels_all)
filenames  = np.array(filenames)

print(f"\n✅ Done!")
print(f"   Slow  : {slow_all.shape}")
print(f"   Fast  : {fast_all.shape}")
print(f"   Labels: {labels_all.shape}")
print(f"   Filenames: {filenames.shape}")
if failed:
    print(f"   ⚠️  Failed: {failed}")

np.save(SAVE_DIR / "slow_frames.npy", slow_all)
np.save(SAVE_DIR / "fast_frames.npy", fast_all)
np.save(SAVE_DIR / "labels.npy",      labels_all)
# ⚠️ Save filenames exactly aligned with the tensors!
np.save(SAVE_DIR / "video_names.npy", filenames)

print(f"\n💾 Saved to: {SAVE_DIR}")
print(f"   slow_frames.npy → {slow_all.nbytes/1e6:.1f} MB")
print(f"   fast_frames.npy → {fast_all.nbytes/1e6:.1f} MB")
print(f"   video_names.npy saved for safe loading later.")

In [ ]:
# ── CELL 11: Verify ───────────────────────────────────────────────────────────

s = np.load(SAVE_DIR / "slow_frames.npy")
f = np.load(SAVE_DIR / "fast_frames.npy")
l = np.load(SAVE_DIR / "labels.npy")
n = np.load(SAVE_DIR / "video_names.npy")

print("✅ Verification passed!")
print(f"   Slow  : {s.shape}  dtype={s.dtype}")
print(f"   Fast  : {f.shape}  dtype={f.dtype}")
print(f"   Labels: {l.shape} — {np.unique(l, return_counts=True)}")
print(f"   Names : {n.shape}")
print("\n🎉 Preprocessing complete and ready to push!")